# 📧 NLP Phishing Email Detection

Bidirectional LSTM + Baseline Models + Feature Engineering + Explainability

In [ ]:
# CELL 1 — Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries loaded ✅")
print(f"TensorFlow version: {tf.__version__}")

## 1. Load & Explore Dataset

In [ ]:
# CELL 2 — Load Dataset
df = pd.read_csv('Phishing_Email.csv')
print("Shape:", df.shape)
print(df.head())
print("
Column names:", df.columns.tolist())

In [ ]:
# CELL 3 — Explore the Data
print("Email counts:")
print(df['Email Type'].value_counts())

plt.figure(figsize=(6,4))
df['Email Type'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Phishing vs Safe Emails')
plt.xlabel('Email Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('class_distribution.png')
plt.show()

print("
Missing values:")
print(df.isnull().sum())

# Drop missing
df = df.dropna(subset=['Email Text', 'Email Type']).reset_index(drop=True)
print(f"
Clean dataset size: {len(df)}")

## 2. Label Encoding & Text Cleaning

In [ ]:
# CELL 4 — Encode Labels
# FIX: map 'Email Type' string labels to binary integers
df['label'] = df['Email Type'].map({'Phishing Email': 1, 'Safe Email': 0})
print("Label distribution:")
print(df['label'].value_counts())
print(df[['Email Type', 'label']].head(5))

## 3. Feature Engineering

We extract hand-crafted features alongside the text to give models extra signal.

In [ ]:
# CELL 5 — Feature Engineering
URGENCY_KEYWORDS = [
    'urgent', 'immediately', 'verify', 'suspend', 'account',
    'click here', 'confirm', 'password', 'bank', 'prize',
    'winner', 'free', 'limited', 'offer', 'expires', 'alert'
]

def extract_features(text):
    text_lower = str(text).lower()
    url_count     = len(re.findall(r'http\S+|www\S+', text_lower))
    email_length  = len(text)
    urgency_count = sum(1 for kw in URGENCY_KEYWORDS if kw in text_lower)
    html_tags     = len(re.findall(r'<[^>]+>', text))
    exclamations  = text.count('!')
    dollar_signs  = text.count('$')
    return pd.Series({
        'url_count'    : url_count,
        'email_length' : email_length,
        'urgency_count': urgency_count,
        'html_tags'    : html_tags,
        'exclamations' : exclamations,
        'dollar_signs' : dollar_signs,
    })

feature_df = df['Email Text'].apply(extract_features)
df = pd.concat([df, feature_df], axis=1)

print("Engineered features added ✅")
print(df[['label','url_count','email_length','urgency_count','html_tags','exclamations','dollar_signs']].groupby('label').mean().round(2))

In [ ]:
# CELL 6 — Visualise Features by Class
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
feature_cols = ['url_count','email_length','urgency_count','html_tags','exclamations','dollar_signs']

for ax, col in zip(axes.flatten(), feature_cols):
    df.groupby('Email Type')[col].mean().plot(kind='bar', ax=ax, color=['steelblue','tomato'])
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Average Feature Values — Phishing vs Safe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_comparison.png')
plt.show()

In [ ]:
# CELL 7 — Text Cleaning
def clean_email(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' url ', text)   # replace URLs
    text = re.sub(r'<.*?>', ' ', text)                      # remove HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)                  # remove symbols/numbers
    text = re.sub(r'\s+', ' ', text).strip()               # collapse whitespace
    return text

df['clean_text'] = df['Email Text'].apply(clean_email)
print("Cleaning done ✅")
print("Sample cleaned email:")
print(df['clean_text'].iloc[3][:200])

## 4. Baseline Models (TF-IDF + Logistic Regression / Naive Bayes)

Before training LSTM, we establish simple baselines so we can measure how much deep learning actually helps.

In [ ]:
# CELL 8 — Train/Test Split (shared across all models)
X_text = df['clean_text'].values
y      = df['label'].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train_text)}  |  Test: {len(X_test_text)}")

In [ ]:
# CELL 9 — Logistic Regression Baseline
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipeline.fit(X_train_text, y_train)
y_pred_lr = lr_pipeline.predict(X_test_text)

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%")
print(classification_report(y_test, y_pred_lr, target_names=['Safe', 'Phishing']))

In [ ]:
# CELL 10 — Naive Bayes Baseline
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)),
    ('clf',   MultinomialNB(alpha=0.1))
])
nb_pipeline.fit(X_train_text, y_train)
y_pred_nb = nb_pipeline.predict(X_test_text)

print("=== Naive Bayes ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_nb)*100:.2f}%")
print(classification_report(y_test, y_pred_nb, target_names=['Safe', 'Phishing']))

## 5. Deep Learning — Bidirectional LSTM

In [ ]:
# CELL 11 — Tokenization & Padding
MAX_WORDS = 50000
MAX_LEN   = 300

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)   # fit ONLY on train to avoid leakage

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, truncating='post', padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, truncating='post', padding='post')

print(f"Vocabulary size : {len(tokenizer.word_index):,}")
print(f"Train shape     : {X_train_pad.shape}")
print(f"Test shape      : {X_test_pad.shape}")

In [ ]:
# CELL 12 — Build Bidirectional LSTM Model
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1,  activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# CELL 13 — Train LSTM
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# CELL 14 — Plot Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train', color='steelblue')
ax1.plot(history.history['val_accuracy'], label='Val',   color='tomato')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history.history['loss'],     label='Train', color='steelblue')
ax2.plot(history.history['val_loss'], label='Val',   color='tomato')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

In [ ]:
# CELL 15 — Evaluate LSTM on Test Set
loss_val, acc_val = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"LSTM Test Accuracy: {acc_val*100:.2f}%")

y_pred_lstm = (model.predict(X_test_pad) > 0.5).astype(int).flatten()

cm = confusion_matrix(y_test, y_pred_lstm)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Safe','Phishing'],
            yticklabels=['Safe','Phishing'])
plt.title('Confusion Matrix — LSTM')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.savefig('confusion_matrix.png')
plt.show()

print("
Classification Report:")
print(classification_report(y_test, y_pred_lstm, target_names=['Safe Email','Phishing Email']))

## 6. Model Comparison

In [ ]:
# CELL 16 — Compare All Models
results = {
    'Logistic Regression': accuracy_score(y_test, y_pred_lr)  * 100,
    'Naive Bayes'        : accuracy_score(y_test, y_pred_nb)  * 100,
    'Bidirectional LSTM' : acc_val * 100,
}

print("Model Accuracy Comparison")
print("-" * 35)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(acc // 2)
    print(f"{name:<22} {acc:.2f}%  {bar}")

plt.figure(figsize=(8,4))
plt.barh(list(results.keys()), list(results.values()),
         color=['#2196F3','#FF5722','#4CAF50'])
plt.xlabel('Accuracy (%)')
plt.title('Model Comparison')
plt.xlim(80, 100)
for i, (k, v) in enumerate(results.items()):
    plt.text(v + 0.1, i, f"{v:.2f}%", va='center')
plt.tight_layout()
plt.savefig('model_comparison.png')
plt.show()

## 7. Explainability — Why Is This Email Phishing?

We use the Logistic Regression TF-IDF model to identify the most important words that triggered a phishing prediction.

In [ ]:
# CELL 17 — Top Phishing / Safe Words (LR Explainability)
tfidf_vec = lr_pipeline.named_steps['tfidf']
lr_model  = lr_pipeline.named_steps['clf']

feature_names = np.array(tfidf_vec.get_feature_names_out())
coefficients  = lr_model.coef_[0]

# Top 15 words pushing toward PHISHING (positive coef) and SAFE (negative coef)
top_phishing_idx = np.argsort(coefficients)[-15:][::-1]
top_safe_idx     = np.argsort(coefficients)[:15]

top_phishing_words = feature_names[top_phishing_idx]
top_phishing_vals  = coefficients[top_phishing_idx]
top_safe_words     = feature_names[top_safe_idx]
top_safe_vals      = coefficients[top_safe_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.barh(top_phishing_words[::-1], top_phishing_vals[::-1], color='tomato')
ax1.set_title('Top 15 Words → PHISHING', fontsize=12, fontweight='bold')
ax1.set_xlabel('LR Coefficient')

ax2.barh(top_safe_words[::-1], np.abs(top_safe_vals[::-1]), color='steelblue')
ax2.set_title('Top 15 Words → SAFE', fontsize=12, fontweight='bold')
ax2.set_xlabel('|LR Coefficient|')

plt.suptitle('Feature Importance — Logistic Regression', fontsize=14)
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

In [ ]:
# CELL 18 — Per-Email Explainability
import re

def explain_prediction(email_text, top_n=8):
    """Highlights which words in an email most influenced the phishing prediction."""
    cleaned = clean_email(email_text)
    
    # Get prediction from LR model
    prob_phishing = lr_pipeline.predict_proba([cleaned])[0][1]
    label = "🚨 PHISHING" if prob_phishing > 0.5 else "✅ SAFE"
    
    # Find contributing words
    tfidf_vec  = lr_pipeline.named_steps['tfidf']
    lr_model   = lr_pipeline.named_steps['clf']
    feat_names = np.array(tfidf_vec.get_feature_names_out())
    coefs      = lr_model.coef_[0]
    
    vec = tfidf_vec.transform([cleaned])
    # Only look at words that appear in this email
    nonzero_idx    = vec.nonzero()[1]
    word_scores    = [(feat_names[i], coefs[i] * vec[0, i]) for i in nonzero_idx]
    word_scores    = sorted(word_scores, key=lambda x: abs(x[1]), reverse=True)[:top_n]
    
    print(f"Email preview : {email_text[:120]}...")
    print(f"Prediction    : {label}  (phishing probability: {prob_phishing:.2%})")
    print(f"
Top contributing words:")
    for word, score in word_scores:
        direction = "→ PHISHING" if score > 0 else "→ SAFE"
        print(f"  {word:<20}  score={score:+.4f}  {direction}")
    print("-" * 60)

# Test on a few emails
print("=== PHISHING SAMPLE ===")
explain_prediction(df[df['label']==1]['Email Text'].iloc[10])

print("
=== SAFE SAMPLE ===")
explain_prediction(df[df['label']==0]['Email Text'].iloc[10])

## 8. Final Demo — Predict Any Email

In [ ]:
# CELL 19 — Demo Function (FIXED: uses )
def predict_email(email_text, actual_label=None):
    """Predict whether a raw email text is phishing or safe."""
    cleaned     = clean_email(email_text)
    seq         = tokenizer.texts_to_sequences([cleaned])
    padded_seq  = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    raw_score   = model.predict(padded_seq, verbose=0)[0][0]

    if raw_score > 0.5:
        label      = "🚨 PHISHING DETECTED"
        confidence = raw_score
    else:
        label      = "✅ SAFE EMAIL"
        confidence = 1 - raw_score

    # FIX: use  so label="SAFE" (falsy check would silently skip it)
    if actual_label is not None:
        print(f"Actual    : {actual_label}")
    print(f"Predicted : {label} ({confidence:.2%} confidence)")
    print(f"Preview   : {email_text[:100]}")
    print("-" * 60)

print("=== PHISHING EMAILS ===")
for i in [5, 10, 15]:
    predict_email(df[df['label']==1]['Email Text'].iloc[i], actual_label="PHISHING")

print("
=== SAFE EMAILS ===")
for i in [5, 10, 15]:
    predict_email(df[df['label']==0]['Email Text'].iloc[i], actual_label="SAFE")

In [ ]:
# CELL 20 — Try Your Own Email
my_email = """
Dear user, your account has been suspended. 
Click here immediately to verify your details and avoid permanent suspension.
Confirm your password at: http://secure-login.fakebank.com
"""

print("Custom email test:")
predict_email(my_email)
print()
explain_prediction(my_email)

## 9. Save Model

In [ ]:
# CELL 21 — Save LSTM Model (use .keras format to avoid deprecation warning)
model.save('lstm_phishing_model.keras')
print("LSTM model saved ✅")

# Save tokenizer
import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Tokenizer saved ✅")

# Save LR pipeline
with open('lr_pipeline.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)
print("LR pipeline saved ✅")